In [1]:
import pandas as pd
import numpy as np
import random
import re
import nltk
from nltk.corpus import wordnet
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

# Download NLTK data with error handling
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', quiet=True)

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet', quiet=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [2]:
train_df = pd.read_csv("Data/train.csv", header=None)
test_df  = pd.read_csv("Data/test.csv", header=None)

train_df.columns = ["label", "title", "review"]
test_df.columns  = ["label", "title", "review"]

train_df["label"] = train_df["label"].astype(int) - 1
test_df["label"]  = test_df["label"].astype(int) - 1

train_df["text"] = train_df["title"] + " " + train_df["review"]
test_df["text"]  = test_df["title"] + " " + test_df["review"]



In [7]:
# Stratified sampling to reduce dataset size (keeps class proportions).
# Adjust TARGET_TRAIN / TARGET_TEST as needed to control runtime.
TARGET_TRAIN = 100000
TARGET_TEST = 20000

# Only down-sample if datasets are larger than the targets
if len(train_df) > TARGET_TRAIN:
    train_df, _ = train_test_split(
        train_df,
        train_size=TARGET_TRAIN,
        stratify=train_df['label'],
        random_state=42
    )

if len(test_df) > TARGET_TEST:
    test_df, _ = train_test_split(
        test_df,
        train_size=TARGET_TEST,
        stratify=test_df['label'],
        random_state=42
    )

print('Sampled Train:', len(train_df))
print('Sampled Test:', len(test_df))


Sampled Train: 100000
Sampled Test: 20000


In [8]:
train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    stratify=train_df["label"],
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

train_df["text"] = train_df["text"].fillna("")
val_df["text"]   = val_df["text"].fillna("")
test_df["text"]  = test_df["text"].fillna("")


Train: 90000
Validation: 10000
Test: 20000


In [9]:
import re
from collections import Counter

# Simple regex tokenizer
def tokenize(text):
    # Handle NaN or non-string values
    if not isinstance(text, str):
        return []
    text = text.lower()
    return re.findall(r"\b\w+\b", text)


# Build vocabulary manually
def build_vocab(texts, min_freq=2):
    counter = Counter()
    
    for text in texts:
        tokens = tokenize(text)
        counter.update(tokens)
    
    vocab = {"<pad>": 0, "<unk>": 1}
    idx = 2
    
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = idx
            idx += 1
    
    return vocab


# Build vocab from training text only
vocab = build_vocab(train_df["text"], min_freq=2)


print("Vocab size:", len(vocab))

Vocab size: 55560


In [17]:
class AmazonDataset(Dataset):
    def __init__(self, df, vocab):
        self.texts = df["text"].tolist()
        self.labels = df["label"].values
        self.vocab = vocab

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        tokens = tokenize(self.texts[idx])
        indices = [self.vocab.get(t, self.vocab["<unk>"]) for t in tokens]
        # Ensure indices are long ints for Embedding layer and labels are long for loss
        return torch.tensor(indices, dtype=torch.long), torch.tensor(self.labels[idx], dtype=torch.long)


In [18]:
def collate_batch(batch):
    texts, labels = zip(*batch)
    texts_pad = pad_sequence(texts, batch_first=True, padding_value=vocab["<pad>"])
    labels = torch.stack(labels)
    return texts_pad, labels


In [19]:
train_loader = DataLoader(AmazonDataset(train_df, vocab), batch_size=64, shuffle=True, collate_fn=collate_batch)
val_loader   = DataLoader(AmazonDataset(val_df, vocab), batch_size=64, collate_fn=collate_batch)
test_loader  = DataLoader(AmazonDataset(test_df, vocab), batch_size=64, collate_fn=collate_batch)


In [20]:
class BiLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, output_dim,
                 emb_dropout=0.2, lstm_dropout=0.2):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.emb_dropout = nn.Dropout(emb_dropout)
        
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=lstm_dropout
        )
        
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        emb = self.embedding(x)
        emb = self.emb_dropout(emb)
        
        out, _ = self.lstm(emb)
        out = out[:, -1, :]
        
        return self.fc(out)


In [21]:
def train_model(model, optimizer, criterion, loader):
    model.train()
    total_loss = 0
    
    for texts, labels in loader:
        texts, labels = texts.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(texts)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    return total_loss / len(loader)


In [22]:
def evaluate_model(model, loader):
    model.eval()
    preds, trues = [], []
    
    with torch.no_grad():
        for texts, labels in loader:
            texts = texts.to(device)
            outputs = model(texts)
            preds.extend(outputs.argmax(1).cpu().numpy())
            trues.extend(labels.numpy())
    
    return f1_score(trues, preds, average="macro")


In [23]:
results = []
dropout_values = [0.2, 0.4, 0.6]

for emb_drop in dropout_values:
    for lstm_drop in dropout_values:
        
        print(f"\nEmbedding Dropout: {emb_drop} | LSTM Dropout: {lstm_drop}")
        
        model = BiLSTM(
            vocab_size=len(vocab),
            embed_dim=128,
            hidden_dim=128,
            output_dim=2,
            emb_dropout=emb_drop,
            lstm_dropout=lstm_drop
        ).to(device)
        
        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss()
        
        for epoch in range(5):
            train_loss = train_model(model, optimizer, criterion, train_loader)
            print(f"Epoch {epoch+1}, Train Loss: {train_loss:.4f}")
        
        val_f1 = evaluate_model(model, val_loader)
        test_f1 = evaluate_model(model, test_loader)
        
        results.append((emb_drop, lstm_drop, train_loss, val_f1, test_f1))
        
        print(f"Validation Macro F1: {val_f1:.4f}")
        print(f"Test Macro F1: {test_f1:.4f}")



Embedding Dropout: 0.2 | LSTM Dropout: 0.2
Epoch 1, Train Loss: 0.6934
Epoch 2, Train Loss: 0.6931
Epoch 3, Train Loss: 0.5733
Epoch 4, Train Loss: 0.2501
Epoch 5, Train Loss: 0.1902
Validation Macro F1: 0.9220
Test Macro F1: 0.9203

Embedding Dropout: 0.2 | LSTM Dropout: 0.4
Epoch 1, Train Loss: 0.6935
Epoch 2, Train Loss: 0.6929
Epoch 3, Train Loss: 0.6797
Epoch 4, Train Loss: 0.5395
Epoch 5, Train Loss: 0.3277
Validation Macro F1: 0.8962
Test Macro F1: 0.8965

Embedding Dropout: 0.2 | LSTM Dropout: 0.6
Epoch 1, Train Loss: 0.6936
Epoch 2, Train Loss: 0.6918
Epoch 3, Train Loss: 0.4952
Epoch 4, Train Loss: 0.2398
Epoch 5, Train Loss: 0.1841
Validation Macro F1: 0.9236
Test Macro F1: 0.9231

Embedding Dropout: 0.4 | LSTM Dropout: 0.2
Epoch 1, Train Loss: 0.6934
Epoch 2, Train Loss: 0.6752
Epoch 3, Train Loss: 0.5928
Epoch 4, Train Loss: 0.3338
Epoch 5, Train Loss: 0.2591
Validation Macro F1: 0.9120
Test Macro F1: 0.9111

Embedding Dropout: 0.4 | LSTM Dropout: 0.4
Epoch 1, Train Loss:

In [24]:
results_df = pd.DataFrame(
    results,
    columns=["Embedding Dropout", "LSTM Dropout", "Train Loss", "Val Macro F1", "Test Macro F1"]
)

print(results_df)


   Embedding Dropout  LSTM Dropout  Train Loss  Val Macro F1  Test Macro F1
0                0.2           0.2    0.190206      0.921994       0.920289
1                0.2           0.4    0.327721      0.896248       0.896498
2                0.2           0.6    0.184091      0.923599       0.923099
3                0.4           0.2    0.259088      0.912000       0.911150
4                0.4           0.4    0.239046      0.920998       0.919048
5                0.4           0.6    0.195015      0.926700       0.925199
6                0.6           0.2    0.289212      0.906748       0.905515
7                0.6           0.4    0.241081      0.920400       0.920399
8                0.6           0.6    0.315914      0.890261       0.893260


In [25]:
def add_spelling_noise(text):
    words = text.split()
    if not words:
        return text
    i = random.randint(0, len(words)-1)
    w = words[i]
    if len(w) > 1:
        pos = random.randint(0, len(w)-1)
        w = w[:pos] + w[pos+1:]
        words[i] = w
    return " ".join(words)


In [26]:
def replace_synonym(text):
    words = text.split()
    if not words:
        return text
    i = random.choice(range(len(words)))
    syns = wordnet.synsets(words[i])
    if syns:
        lemmas = syns[0].lemmas()
        if len(lemmas) > 1:
            words[i] = lemmas[1].name()
    return " ".join(words)


In [27]:
noisy_texts = []
noisy_labels = []

for text, label in zip(test_df["text"], test_df["label"]):
    if random.random() < 0.5:
        noisy = add_spelling_noise(text)
    else:
        noisy = replace_synonym(text)
    noisy_texts.append(noisy)
    noisy_labels.append(label)

noisy_df = pd.DataFrame({"text": noisy_texts, "label": noisy_labels})
noisy_loader = DataLoader(AmazonDataset(noisy_df, vocab), batch_size=64, collate_fn=collate_batch)


In [28]:
noisy_f1 = evaluate_model(model, noisy_loader)
print("Macro F1 on Noisy Test Data:", noisy_f1)


Macro F1 on Noisy Test Data: 0.8900574983978741
